# Databricks to Neon PostgreSQL Setup

This notebook connects Databricks to a PostgreSQL database hosted on Neon and transfers the final project results to the database.

### Connection flow

Databricks  
↓  
Secret Scope `neon`  
↓  
Stored PostgreSQL credentials  
↓  
Neon PostgreSQL

The password should remain stored securely in Databricks Secrets and should not be written directly in the notebook.

##1. Neon PostgreSQL Connection

In [0]:
# Neon PostgreSQL connection information

neon_host = "ep-summer-sea-ag9wnzfy-pooler.c-2.eu-central-1.aws.neon.tech"
neon_port = "5432"
neon_database = "neondb"
neon_user = "neondb_owner"

# Replace this with your current Neon password
neon_password = dbutils.secrets.get(
    scope="neon",
    key="password"
)
jdbc_url = (
    f"jdbc:postgresql://{neon_host}:{neon_port}/"
    f"{neon_database}?sslmode=require"
)

print("Neon connection parameters created successfully")

Neon connection parameters created successfully


##2. Test the Connection Between Databricks and Neon

In [0]:
connection_test_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        """
        SELECT
            current_database() AS database_name,
            current_user AS connected_user,
            version() AS postgres_version
        """
    )
    .option("user", neon_user)
    .option("password", neon_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(connection_test_df)

database_name,connected_user,postgres_version
neondb,neondb_owner,"PostgreSQL 18.4 (be2730e) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit"


##3. Display the Available Databricks Analysis Tables

In [0]:
analysis_tables_df = spark.sql(
    "SHOW TABLES IN workspace.analysis_data"
)

display(analysis_tables_df)

database,tableName,isTemporary
analysis_data,aisle_kpis,false
analysis_data,customer_kpis,false
analysis_data,daily_kpis,false
analysis_data,department_kpis,false
analysis_data,hourly_kpis,false
analysis_data,order_kpis,false
analysis_data,product_kpis,false
analysis_data,time_period_kpis,false


##4. Load the Order KPI Table

In [0]:
order_kpis_df = spark.table(
    "workspace.analysis_data.order_kpis"
)

display(order_kpis_df.limit(10))

order_id,user_id,eval_set,order_set,order_number,order_dow,order_hour_of_day,order_time_period,days_since_prior_order,is_first_order,is_final_order,basket_size,distinct_product_count,reordered_product_count,new_product_count,reorder_rate,_analysis_processed_at
100,62781,prior,prior,1,2,13,afternoon,null,true,false,2,2,0,2,0.0,2026-07-27T00:33:44.812Z
138,197422,prior,prior,20,0,16,afternoon,7.0,false,false,6,6,5,1,0.8333,2026-07-27T00:33:44.812Z
250,34840,prior,prior,13,1,7,morning,17.0,false,false,2,2,2,0,1.0,2026-07-27T00:33:44.812Z
265,139432,prior,prior,38,0,9,morning,9.0,false,false,36,36,25,11,0.6944,2026-07-27T00:33:44.812Z
376,121858,prior,prior,11,5,16,afternoon,14.0,false,false,7,7,6,1,0.8571,2026-07-27T00:33:44.812Z
477,13665,prior,prior,3,4,12,afternoon,30.0,false,false,2,2,0,2,0.0,2026-07-27T00:33:44.812Z
522,35250,prior,prior,6,1,15,afternoon,12.0,false,false,20,20,10,10,0.5,2026-07-27T00:33:44.812Z
545,35519,prior,prior,7,0,9,morning,1.0,false,false,2,2,0,2,0.0,2026-07-27T00:33:44.812Z
656,201665,prior,prior,1,3,14,afternoon,null,true,false,5,5,0,5,0.0,2026-07-27T00:33:44.812Z
666,5152,prior,prior,2,5,12,afternoon,13.0,false,false,7,7,5,2,0.7143,2026-07-27T00:33:44.812Z


##6. Export All Analysis Tables to Neon

In [0]:
# 6. Export selected analysis tables to Neon

tables_to_export = [
    "aisle_kpis",
    "customer_kpis",
    "daily_kpis",
    "department_kpis",
    "hourly_kpis",
    "product_kpis",
    "time_period_kpis"
]

print("Tables to export:", tables_to_export)

for table_name in tables_to_export:

    source_table = f"workspace.analysis_data.`{table_name}`"
    target_table = f"public.{table_name}"

    print(f"Writing {source_table} -> {target_table}")

    try:
        (
            spark.table(source_table)
            .write
            .format("postgresql")
            .option("host", neon_host)
            .option("port", neon_port)
            .option("database", neon_database)
            .option("dbtable", target_table)
            .option("user", neon_user)
            .option("password", neon_password)
            .option("batchsize", "1000")
            .option("connectTimeout", "60")
            .mode("overwrite")
            .save()
        )

        print(f"Successfully exported: {target_table}")

    except Exception as error:
        print(f"Failed to export: {source_table}")
        print(str(error))

Tables to export: ['aisle_kpis', 'customer_kpis', 'daily_kpis', 'department_kpis', 'hourly_kpis', 'product_kpis', 'time_period_kpis']
Writing workspace.analysis_data.`aisle_kpis` -> public.aisle_kpis
Failed to export: workspace.analysis_data.`aisle_kpis`
name 'neon_host' is not defined
Writing workspace.analysis_data.`customer_kpis` -> public.customer_kpis
Failed to export: workspace.analysis_data.`customer_kpis`
name 'neon_host' is not defined
Writing workspace.analysis_data.`daily_kpis` -> public.daily_kpis
Failed to export: workspace.analysis_data.`daily_kpis`
name 'neon_host' is not defined
Writing workspace.analysis_data.`department_kpis` -> public.department_kpis
Failed to export: workspace.analysis_data.`department_kpis`
name 'neon_host' is not defined
Writing workspace.analysis_data.`hourly_kpis` -> public.hourly_kpis
Failed to export: workspace.analysis_data.`hourly_kpis`
name 'neon_host' is not defined
Writing workspace.analysis_data.`product_kpis` -> public.product_kpis
Fail

##7. Verify the Tables Created in Neon

In [0]:
neon_tables_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        """
        SELECT
            table_schema,
            table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
        """
    )
    .option("user", neon_user)
    .option("password", neon_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(neon_tables_df)

table_schema,table_name
public,aisle_kpis
public,customer_kpis
public,daily_kpis
public,department_kpis
public,hourly_kpis
public,order_kpis
public,product_kpis
public,time_period_kpis


##8. Read a Neon Table Back into Databricks

In [0]:
neon_order_kpis_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.order_kpis")
    .option("user", neon_user)
    .option("password", neon_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(neon_order_kpis_df.limit(10))

order_id,user_id,eval_set,order_set,order_number,order_dow,order_hour_of_day,order_time_period,days_since_prior_order,is_first_order,is_final_order,basket_size,distinct_product_count,reordered_product_count,new_product_count,reorder_rate,_analysis_processed_at
95,187742,prior,prior,39,5,7,morning,5.0,false,false,14,14,12,2,0.8571,2026-07-27T00:33:44.812Z
750,71771,prior,prior,34,5,10,morning,8.0,false,false,24,24,21,3,0.875,2026-07-27T00:33:44.812Z
1153,73225,prior,prior,1,6,11,morning,null,true,false,7,7,0,7,0.0,2026-07-27T00:33:44.812Z
1171,100330,prior,prior,20,1,13,afternoon,11.0,false,false,61,61,48,13,0.7869,2026-07-27T00:33:44.812Z
2160,154317,prior,prior,6,1,15,afternoon,8.0,false,false,19,19,3,16,0.1579,2026-07-27T00:33:44.812Z
2615,107242,prior,prior,2,0,12,afternoon,4.0,false,false,3,3,1,2,0.3333,2026-07-27T00:33:44.812Z
2643,162181,prior,prior,14,0,15,afternoon,3.0,false,false,5,5,5,0,1.0,2026-07-27T00:33:44.812Z
2697,127410,prior,prior,3,6,13,afternoon,6.0,false,false,16,16,3,13,0.1875,2026-07-27T00:33:44.812Z
2756,179342,prior,prior,51,5,11,morning,1.0,false,false,9,9,7,2,0.7778,2026-07-27T00:33:44.812Z
2847,143196,prior,prior,4,6,14,afternoon,13.0,false,false,13,13,10,3,0.7692,2026-07-27T00:33:44.812Z


##9. Check the Number of Rows in Neon

In [0]:
order_kpis_count_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        """
        SELECT COUNT(*) AS number_of_rows
        FROM public.order_kpis
        """
    )
    .option("user", neon_user)
    .option("password", neon_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(order_kpis_count_df)

number_of_rows
3346083
